In [ ]:
from qiskit import QuantumCircuit , QuantumRegister ,ClassicalRegister

In [ ]:
# f(x) values
# change the output strings and run all cells again
# input order is q[2]q[1]q[0]
# output order is q[5]q[4]q[3]
fx = {
    '000': '000',
    '001': '001',
    '010': '010',
    '011': '011',
    '100': '010',
    '101': '011',
    '110': '000',
    '111': '001'
}

# each output must occur exactly twice
# every matching pair must have the same nonzero x xor x'
# the sample table has secret 110; it is not hardcoded in Uf

In [ ]:
# check the two-to-one Simon promise
inputs = [format(i,'03b') for i in range(8)]

if set(fx) != set(inputs):
    raise ValueError("give f(x) for all inputs from 000 to 111")

pairs = {}
for i in fx:
    value = fx[i]
    if not isinstance(value,str) or len(value) != 3 or set(value) - {'0','1'}:
        raise ValueError("each output should be a 3 bit string")
    if value not in pairs:
        pairs[value] = []
    pairs[value].append(i)

periods = set()
for value in pairs:
    if len(pairs[value]) != 2:
        raise ValueError("each output should occur exactly twice")
    a,b = pairs[value]
    periods.add(int(a,2) ^ int(b,2))

if len(periods) != 1 or 0 in periods:
    raise ValueError("all matching pairs should have the same nonzero xor")

# this only validates the supplied table
# the answer below is recovered from measured equations

In [ ]:
q = QuantumRegister(6,'q')
c = ClassicalRegister(3,'c')

In [ ]:
qc = QuantumCircuit(q,c)

In [ ]:
#input preparing
# q[0], q[1], q[2] are input qubits
# q[3], q[4], q[5] are output qubits
qc.h(q[0])
qc.h(q[1])
qc.h(q[2])

qc.barrier()

In [ ]:
#oracle
# Uf |x>|y> = |x>|y xor f(x)>
# truth table construction for this small 3 input qubit example
uf = QuantumCircuit(6, name='Uf')

for i in fx:
    bits = i[::-1]
    output = fx[i][::-1]

    # select this input row
    for j in range(3):
        if bits[j] == '0':
            uf.x(j)

    # xor each output bit with f(x)
    for j in range(3):
        if output[j] == '1':
            uf.mcx([0,1,2],3+j)

    # restore the input qubits
    for j in range(3):
        if bits[j] == '0':
            uf.x(j)

uf.draw('mpl')

In [ ]:
# apply Uf to our circuit
qc.compose(uf, inplace=True)
qc.barrier()
qc.draw('mpl')

In [ ]:
qc.h(q[0])
qc.h(q[1])
qc.h(q[2])

qc.barrier()
qc.measure(q[0],c[0])
qc.measure(q[1],c[1])
qc.measure(q[2],c[2])

qc.draw('mpl')

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile

# Create simulator
simulator = AerSimulator()

# Convert the oracle gates into gates supported by the simulator
compiled_qc = transpile(qc, simulator)
job = simulator.run(compiled_qc, shots=1000)

# Get result
result = job.result()
# Get counts
counts = result.get_counts()

print(counts)

In [ ]:
# each measured string y gives y dot s = 0 modulo 2
# counts are displayed in c[2]c[1]c[0] order
for i in counts:
    print(i,"probability: ",counts[i]/1000)

In [ ]:
# try the nonzero 3 bit strings and check all measured equations
# this simple search is only for this small 3 bit example
# larger examples use elimination modulo 2
possible = []

for number in range(1,8):
    s = format(number,'03b')
    correct = True

    for i in counts:
        value = 0
        for j in range(3):
            value += int(i[j])*int(s[j])

        if value % 2 != 0:
            correct = False
            break

    if correct:
        possible.append(s)

if len(possible) == 1:
    print("secret string: ",possible[0])
else:
    print("possible strings: ",possible)
    print("need more independent measurements, run simulator again")